# Atlas integrated scoring

This notebook calculates Levels 1 and 2 of the **Atlas Integrated Structure** for every BSR represented in the five input CSVs.

| Level | Main question | Principal outputs |
|---|---|---|
| **1. Integrated risk** | Where do fish use, limiting-factor condition, biological vulnerability, and population priority overlap? | Life-stage impact and risk, limiting-factor impact and risk, and overall BSR risk |
| **2. Action benefit** | Which action types address the limiting factors contributing most to calculated risk? | Condition improvement, limiting-factor amelioration, and overall action benefit by BSR and action type |

The notebook uses the scores already provided in the standardized CSVs. It does not recreate the source workbooks. The BSR-level `fish_use_score_decimal` is used for overall limiting-factor impact, while the life-stage `LS_corrected_score` is used for population-weighted risk. The species-level `species_aggregate_score` is retained for reporting. The notebook also uses the continuous 0.1-to-1 vulnerability score and the raw LFAT score, defined as directness multiplied by frequency. Level 3 project scoring is not included.

The seven core score CSVs and `bsr_scores.gpkg` are written to `data/outputs`. The GeoPackage includes the fish-use and population inputs as attribute tables in addition to the calculated score tables. Supporting review and quality-control tables are written to `data/outputs/QC`.


## 1. Inputs, outputs, and the only routine setting

Place the following files in `data/inputs`:

- `Fish Use Scores.csv`
- `LFAT.csv`
- `Limiting factor scores.csv`
- `Population scores.csv`
- `Vulnerability table.csv`
- `bsr.gpkg`

The automatic file search also accepts a single parenthetically numbered copy of each input, such as `Fish Use Scores(8).csv` or `bsr(1).gpkg`. If needed, set `INPUT_DIR_OVERRIDE` below to the folder containing the inputs. No other settings are normally required.


In [ ]:
from contextlib import closing
from pathlib import Path
from uuid import uuid4
import hashlib
import shutil
import sqlite3

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if isinstance(obj, pd.DataFrame):
            print(obj.to_string(index=False))
        else:
            print(obj)


# Optional: replace None with a folder path if automatic discovery is not appropriate.
INPUT_DIR_OVERRIDE = None

INPUT_STEMS = {
    "fish_use": "Fish Use Scores",
    "lfat": "LFAT",
    "limiting_factor": "Limiting factor scores",
    "population": "Population scores",
    "vulnerability": "Vulnerability table",
}
FISH_USE_COLUMNS = [
    "bsr", "basin", "bsr_crosswalk_status",
    "species", "life_stage", "LS_corrected_score",
    "species_aggregate_score", "fish_use_score_decimal",
]
BSR_INPUT_FILE = "bsr.gpkg"
BSR_OUTPUT_FILE = "bsr_scores.gpkg"


def select_input_file(folder, stem, suffix=".csv"):
    # Return one exact or parenthetically numbered input, or None.
    exact = folder / f"{stem}{suffix}"
    if exact.exists():
        return exact
    matches = sorted(folder.glob(f"{stem}(*){suffix}"))
    return matches[0] if len(matches) == 1 else None


def candidate_input_directories(start):
    seen = set()
    for folder in (start, *start.parents):
        for candidate in (
            folder / "data" / "inputs",
            folder / "upload",
            folder,
        ):
            resolved = candidate.resolve()
            if resolved not in seen:
                seen.add(resolved)
                yield resolved


def locate_inputs(start, override=None):
    candidates = [Path(override).expanduser().resolve()] if override else list(
        candidate_input_directories(start)
    )
    for folder in candidates:
        if not folder.is_dir():
            continue
        selected = {
            key: select_input_file(folder, stem)
            for key, stem in INPUT_STEMS.items()
        }
        spatial_path = select_input_file(folder, "bsr", ".gpkg")
        if (
            all(path is not None for path in selected.values())
            and spatial_path is not None
        ):
            return folder, selected, spatial_path
    expected = [
        *(f"{stem}.csv" for stem in INPUT_STEMS.values()),
        BSR_INPUT_FILE,
    ]
    raise FileNotFoundError(
        "Could not find one complete input set. Expected: "
        + ", ".join(expected)
    )


INPUT_DIR, INPUT_PATHS, BSR_INPUT_PATH = locate_inputs(
    Path.cwd().resolve(), INPUT_DIR_OVERRIDE
)
if INPUT_DIR.name == "inputs" and INPUT_DIR.parent.name == "data":
    REPO_ROOT = INPUT_DIR.parent.parent
else:
    REPO_ROOT = Path.cwd().resolve()

OUTPUT_DIR = REPO_ROOT / "data" / "outputs"
QC_DIR = OUTPUT_DIR / "QC"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)
BSR_OUTPUT_PATH = OUTPUT_DIR / BSR_OUTPUT_FILE

raw = {
    key: pd.read_csv(
        path, usecols=FISH_USE_COLUMNS if key == "fish_use" else None
    )
    for key, path in INPUT_PATHS.items()
}

input_summary = pd.DataFrame(
    [
        {
            "dataset": key,
            "file": path.name,
            "rows": len(raw[key]),
            "columns": len(raw[key].columns),
        }
        for key, path in INPUT_PATHS.items()
    ]
)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"QC directory: {QC_DIR}")
print(f"Spatial input: {BSR_INPUT_PATH.name}")
display(input_summary)


## 2. Score fields and transformations

Higher values increase the calculated score. The inputs are used as follows:

| Component | Source field | Scale used | Treatment in this notebook |
|---|---|---:|---|
| Life-stage fish use | `LS_corrected_score` | 0 to 1 for normalization | Used without transformation as the fish-use multiplier in population-weighted risk; QC verifies the configured theoretical maximum |
| Species fish use | `species_aggregate_score` | Source scale | Carried without transformation for reporting and quality control |
| BSR fish use | `fish_use_score_decimal` | 0 to 1 | Used without transformation as the fish-use multiplier in overall limiting-factor impact |
| Population priority | `population_priority` | 0 to 1 | Used without transformation; priorities sum to approximately 1 within each basin and species |
| Limiting-factor condition | `lf_condition_score` | 0.1 to 1 | Used as supplied; the CSV maps its raw 1-to-5 score linearly to 0.1-to-1 |
| Vulnerability | `vulnerability_score` | 0.1 to 1 | Used as supplied; rank 1 maps to 1.0 and rank 15 maps to 0.1 |
| Action relationship | `lfat_score` | 0.01 to 1 | Used as supplied; equals `directness_value × frequency_value` |

The detailed outputs retain `LS_corrected_score` and `species_aggregate_score` under their source field names. `fish_use_score_decimal` is renamed to `fish_use_score` inside the notebook to keep the integrated-score schemas stable. Overall limiting-factor impact and population-weighted risk deliberately use different fish-use inputs: the former uses the BSR-level score, while the latter uses the applicable life-stage score. This prevents a species or life stage with zero fish use from receiving positive risk solely because the BSR has fish use from other species or stages.

### Normalized score convention

All original score fields are retained unchanged. Parallel fields ending in `_normalized` express impact, risk, condition improvement, limiting-factor amelioration, and benefit as contributions to a theoretical 0-to-1 total. The notebook does **not** min-max scale against the highest observed BSR. Instead, it divides raw scores by fixed score-framework maxima derived from the number of modeled relationships, the supplied population-priority weights, and the complete LFAT crosswalk.

`LIFE_STAGE_FISH_USE_MAX` is explicitly configured as `1.0`. QC fails if `LS_corrected_score` falls outside 0 to this fixed upper bound. If the source scoring framework documents a different theoretical maximum, change that constant rather than deriving the maximum from the current dataset.

Because all child scores within a score family use the same denominator, the normalized components remain additive and traceable to the original scores.


In [ ]:
EXPECTED_COLUMNS = {
    "fish_use": FISH_USE_COLUMNS,
    "lfat": [
        "action_id", "action_type", "action_definition",
        "source_action_label", "limiting_factor",
        "limiting_factor_occurrence", "directness_code",
        "directness_rating", "directness_value", "frequency_code",
        "frequency_rating", "frequency_value", "lfat_score",
        "source_sheet", "source_row", "source_directness_cell",
        "source_frequency_cell", "source_score_cell", "source_notes_cell",
    ],
    "limiting_factor": [
        "bsr", "limiting_factor", "n", "mean_r", "median_r",
        "geo_mean_r", "sd_r", "var_r", "iqr_r", "min_r", "max_r",
        "range_r", "flag_spread", "flag_low_n", "any_flag",
        "lf_condition_score_raw_1_5", "lf_condition_score",
        "condition_transformation",
    ],
    "population": [
        "basin", "species", "life_stage", "population_priority",
    ],
    "vulnerability": [
        "species", "source_life_stage", "life_stage", "limiting_factor",
        "vulnerability_rank", "vulnerability_score", "uncertainty_flag",
        "review_flag", "review_reason", "source_sheet",
        "source_rank_cell", "source_rating_cell", "source_notes_cell",
        "source_uncertainty_cell",
    ],
}

CANONICAL_LF = [
    "Anthropogenic Barriers",
    "Riparian Condition",
    "Floodplain Condition",
    "Side Channel and Wetland Habitat",
    "Channel and Habitat Structure",
    "Decreased Water Quantity",
    "Altered Flow Timing",
    "Decreased Sediment Quantity",
    "Increased Sediment Quantity",
    "Summer Water Temperature",
    "Winter Water Temperature",
    "Water Quality",
    "Predation",
    "Altered Primary Productivity",
    "Non-Native Species Interactions and Competition",
]

schema_rows = []
for key, table in raw.items():
    observed = table.columns.tolist()
    expected = EXPECTED_COLUMNS[key]
    schema_rows.append(
        {
            "dataset": key,
            "observed_columns": len(observed),
            "expected_columns": len(expected),
            "schema_pass": observed == expected,
        }
    )
schema_qc = pd.DataFrame(schema_rows)
if not schema_qc["schema_pass"].all():
    details = {
        key: {
            "observed": raw[key].columns.tolist(),
            "expected": EXPECTED_COLUMNS[key],
        }
        for key in raw
        if raw[key].columns.tolist() != EXPECTED_COLUMNS[key]
    }
    raise ValueError(f"One or more input schemas do not match: {details}")

fish_use = raw["fish_use"].rename(
    columns={"fish_use_score_decimal": "fish_use_score"}
).copy()
population = raw["population"].copy()
condition = raw["limiting_factor"].rename(
    columns={
        "lf_condition_score_raw_1_5": "condition_score_raw_1_5",
        "lf_condition_score": "condition_score",
    }
).copy()
vulnerability = raw["vulnerability"].copy()
lfat = raw["lfat"].copy()

assumptions = pd.DataFrame(
    [
        ["Life-stage fish use", "LS_corrected_score", "Used as supplied in population-weighted risk and retained in detailed outputs."],
        ["Species fish use", "species_aggregate_score", "Retained under its source field name for reporting and quality control."],
        ["BSR fish use", "fish_use_score_decimal", "BSR-level 0-to-1 aggregate used as supplied in overall limiting-factor impact."],
        ["Population priority", "population_priority", "Retained as a decimal proportion."],
        ["Limiting-factor condition", "condition_score", "Source 1-to-5 score already mapped to 0.1-to-1 in the CSV."],
        ["Vulnerability", "vulnerability_score", "Rank 1-to-15 already mapped to 1.0-to-0.1 in the CSV."],
        ["Combined migration", "maximum vulnerability_score", "Uses the higher adult or juvenile migration value without counting migration twice."],
        ["LFAT", "lfat_score", "Raw directness multiplied by frequency; no D/I/N replacement."],
        ["Normalization", "LIFE_STAGE_FISH_USE_MAX", "Fixed theoretical upper bound of 1.0 for LS_corrected_score; QC fails rather than deriving the bound from observed scores."],
        ["Level 1 normalization", "*_normalized", "Raw impact and risk are divided by common BSR-level theoretical maxima so detailed normalized contributions still sum to normalized BSR totals."],
        ["Level 2 normalization", "*_normalized", "Raw action scores are divided by maxima based on the complete LFAT crosswalk so normalized action contributions still sum to normalized overall action scores."],
        ["BSR identifiers", "bsr", "The fish-use and limiting-factor files use the same CC1/CC2/etc. identifiers."],
    ],
    columns=["component", "field_or_rule", "implementation"],
)

display(schema_qc)
display(assumptions)


## 3. Align vulnerability life stages

The vulnerability table contains separate `Adult Migration & Holding` and `Juvenile Emigration` rows, while the fish-use and population tables contain one combined `Migration` life stage. The notebook therefore collapses the two source rows to one species × life stage × limiting-factor relationship.

For a combined migration relationship, the value used is:

$$
\begin{aligned}
\text{combined migration vulnerability}
={}&\max\Bigl(
\text{adult migration vulnerability},\\
&\qquad \text{juvenile migration vulnerability}
\Bigr)
\end{aligned}
$$

The calculation is applied separately for each species and limiting factor using the continuous 0.1-to-1 vulnerability score. Taking the maximum represents the more vulnerable migration pathway and avoids adding adult and juvenile migration as if they were two independent fish-use stages. All other life stages have one source row, so their values pass through unchanged.


In [ ]:
def any_yes(values):
    is_yes = (
        values.fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("yes")
    )
    return "Yes" if is_yes.any() else "No"


vulnerability_collapsed = (
    vulnerability.groupby(
        ["species", "life_stage", "limiting_factor"], as_index=False
    )
    .agg(
        vulnerability_rank_min=("vulnerability_rank", "min"),
        vulnerability_rank_max=("vulnerability_rank", "max"),
        vulnerability_score_min=("vulnerability_score", "min"),
        vulnerability_score_max=("vulnerability_score", "max"),
        vulnerability_score=("vulnerability_score", "max"),
        source_vulnerability_rows=("source_life_stage", "size"),
        source_life_stages=(
            "source_life_stage",
            lambda values: " | ".join(sorted(set(values.astype(str)))),
        ),
        vulnerability_review_flag=("review_flag", any_yes),
        uncertainty_notes=(
            "uncertainty_flag",
            lambda values: " | ".join(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
    )
)

display(vulnerability_collapsed.head(10))


## 4. Calculate Level 1 row-level impact and risk

Each fish-use row is expanded across the 15 limiting factors. The calculation unit is one BSR × species × life stage × limiting factor combination.

### Calculation inputs

| Display term | Notebook field | Use in Level 1 |
|---|---|---|
| Overall fish use | `fish_use_score` | BSR-level 0-to-1 multiplier used in impact |
| Life-stage fish use | `LS_corrected_score` | Species- and life-stage-specific multiplier used in risk |
| Condition | `condition_score` | BSR- and limiting-factor-specific 0.1-to-1 multiplier |
| Vulnerability | `vulnerability_score` | Species-, life-stage-, and limiting-factor-specific 0.1-to-1 multiplier |
| Population priority | `population_priority` | Basin-, species-, and life-stage-specific 0-to-1 multiplier used in risk |

The raw row-level impact component is:

$$
\text{impact component}
=
\text{overall fish use}
\times \text{condition}
\times \text{vulnerability}
$$

The raw row-level population-weighted risk component uses the applicable life-stage fish-use score and population priority:

$$
\begin{aligned}
\text{risk component}
={}&\text{life-stage fish use}
\times \text{condition}\\
&{}\times \text{vulnerability}
\times \text{population priority}
\end{aligned}
$$

The impact component is zero when overall BSR fish use is zero. The risk component is zero when the applicable life-stage fish-use score is zero, even if other species or life stages produce a positive overall BSR fish-use score. Otherwise, each component increases when any contributing multiplier increases. These are prioritization components, not probabilities.

### Level 1 normalization

Let:

- $F$ = number of limiting factors, fixed at 15;
- $J_b$ = number of species × life-stage combinations represented in BSR $b$;
- $P_b$ = sum of the supplied `population_priority` weights across all modeled species and life stages in the basin containing BSR $b$; and
- $M_{LS}$ = theoretical maximum life-stage fish-use score, configured as `LIFE_STAGE_FISH_USE_MAX = 1.0`.

The maximum possible raw Level 1 totals for BSR $b$ are:

$$
D_{I,b}=J_b\times F
$$

$$
D_{R,b}=M_{LS}\times P_b\times F
$$

When population priorities sum to 1 within each species, $P_b$ equals the number of modeled species. The notebook uses the supplied priority weights directly so small source rounding differences remain traceable.

The normalized row-level components are:

$$
\text{impact component}_{normalized}
=
\frac{\text{impact component}}{D_{I,b}}
$$

$$
\text{risk component}_{normalized}
=
\frac{\text{risk component}}{D_{R,b}}
$$

These normalized row components are contributions to the **overall** 0-to-1 impact or risk scale. They are not independently stretched to fill 0 to 1. Summing them therefore reproduces the normalized aggregate scores exactly.


In [ ]:
fish_population = fish_use.merge(
    population[["basin", "species", "life_stage", "population_priority"]],
    on=["basin", "species", "life_stage"],
    how="left",
    validate="many_to_one",
)

calculation_grid = (
    fish_population.merge(
        vulnerability_collapsed,
        on=["species", "life_stage"],
        how="left",
        validate="many_to_many",
    )
    .merge(
        condition[
            [
                "bsr", "limiting_factor", "condition_score_raw_1_5",
                "condition_score", "n", "any_flag",
            ]
        ],
        on=["bsr", "limiting_factor"],
        how="left",
        validate="many_to_one",
    )
)

# Normalization uses theoretical maxima from the fixed scoring structure,
# never the maximum observed score in the current BSR dataset.
LIFE_STAGE_FISH_USE_MAX = 1.0
N_LIMITING_FACTORS = len(CANONICAL_LF)
LFAT_TOTAL_WEIGHT = float(lfat["lfat_score"].sum())

bsr_structure = (
    fish_use[["bsr", "basin", "species", "life_stage"]]
    .drop_duplicates()
    .groupby(["bsr", "basin"], as_index=False)
    .agg(species_life_stage_count=("life_stage", "size"))
)
population_weight_totals = (
    population.groupby("basin", as_index=False)
    .agg(population_priority_total=("population_priority", "sum"))
)
normalization_denominators = bsr_structure.merge(
    population_weight_totals,
    on="basin",
    how="left",
    validate="many_to_one",
)
normalization_denominators["impact_normalization_denominator"] = (
    normalization_denominators["species_life_stage_count"]
    * N_LIMITING_FACTORS
)
normalization_denominators["risk_normalization_denominator"] = (
    normalization_denominators["population_priority_total"]
    * LIFE_STAGE_FISH_USE_MAX
    * N_LIMITING_FACTORS
)
normalization_denominators[
    "condition_improvement_normalization_denominator"
] = LFAT_TOTAL_WEIGHT
normalization_denominators[
    "amelioration_normalization_denominator"
] = (
    normalization_denominators["species_life_stage_count"]
    * LFAT_TOTAL_WEIGHT
)
normalization_denominators["benefit_normalization_denominator"] = (
    normalization_denominators["population_priority_total"]
    * LIFE_STAGE_FISH_USE_MAX
    * LFAT_TOTAL_WEIGHT
)

calculation_grid = calculation_grid.merge(
    normalization_denominators[
        [
            "bsr", "basin",
            "impact_normalization_denominator",
            "risk_normalization_denominator",
        ]
    ],
    on=["bsr", "basin"],
    how="left",
    validate="many_to_one",
)

# Original raw score fields are retained unchanged.
calculation_grid["impact_component"] = (
    calculation_grid["fish_use_score"]
    * calculation_grid["condition_score"]
    * calculation_grid["vulnerability_score"]
)
calculation_grid["risk_component"] = (
    calculation_grid["LS_corrected_score"]
    * calculation_grid["condition_score"]
    * calculation_grid["vulnerability_score"]
    * calculation_grid["population_priority"]
)

# Parallel normalized fields use the BSR-level theoretical maxima. Using one
# denominator per score family preserves exact additivity through aggregation.
calculation_grid["impact_component_normalized"] = (
    calculation_grid["impact_component"]
    / calculation_grid["impact_normalization_denominator"]
)
calculation_grid["risk_component_normalized"] = (
    calculation_grid["risk_component"]
    / calculation_grid["risk_normalization_denominator"]
)

display(
    calculation_grid[
        [
            "bsr", "species", "life_stage", "limiting_factor",
            "LS_corrected_score", "species_aggregate_score",
            "fish_use_score", "population_priority",
            "condition_score_raw_1_5", "condition_score",
            "vulnerability_rank_min", "vulnerability_rank_max",
            "vulnerability_score", "impact_component", "risk_component",
            "impact_component_normalized", "risk_component_normalized",
        ]
    ].head(10)
)


## 5. Aggregate Level 1 scores

The same row-level grid is summarized in three ways. No additional weights or averages are introduced. The original raw score fields remain unchanged, and each summary also includes a parallel `_normalized` field.

The normalized life-stage and limiting-factor scores use the same BSR-level denominators $D_{I,b}$ and $D_{R,b}$ defined above. This is important: they represent contributions to the same overall 0-to-1 total, so they still add exactly to the BSR total.

### Species and life-stage impact and risk summaries

For each BSR, species, and life stage, the impact and risk components are summed across the 15 limiting factors:

$$
\begin{aligned}
\text{life-stage impact score}
={}&
\sum_{f=1}^{15}
\left(
\text{overall fish use}
\times \text{condition}_f
\times \text{vulnerability}_f
\right)
\end{aligned}
$$

$$
\text{life-stage impact score}_{normalized}
=
\frac{\text{life-stage impact score}}{D_{I,b}}
=
\sum_{f=1}^{15}\text{impact component}_{normalized}
$$

$$
\begin{aligned}
\text{life-stage risk score}
={}&
\sum_{f=1}^{15}
\left(
\text{life-stage fish use}
\times \text{condition}_f
\times \text{vulnerability}_f
\times \text{population priority}
\right)
\end{aligned}
$$

$$
\text{life-stage risk score}_{normalized}
=
\frac{\text{life-stage risk score}}{D_{R,b}}
=
\sum_{f=1}^{15}\text{risk component}_{normalized}
$$

The highest-priority life-stage indicator is based on the largest raw species and life-stage risk score within each BSR:

$$
\text{risk score for highest priority life stage}
=
\max_{s,l}\left(\text{life-stage risk score}_{s,l}\right)
$$

Because every life-stage risk score in a BSR uses the same denominator, ranking the normalized scores gives the same result. `top_species_life_stage_risk_score_normalized` reports the top contribution on the common 0-to-1 scale.

### Limiting-factor impact and risk summaries

For each BSR and limiting factor:

$$
\text{limiting-factor impact}_f
=
\sum_{s,l}\text{impact component}_{s,l,f}
$$

$$
\text{limiting-factor impact}_{f,normalized}
=
\frac{\text{limiting-factor impact}_f}{D_{I,b}}
=
\sum_{s,l}\text{impact component}_{s,l,f,normalized}
$$

$$
\text{limiting-factor risk}_f
=
\sum_{s,l}\text{risk component}_{s,l,f}
$$

$$
\text{limiting-factor risk}_{f,normalized}
=
\frac{\text{limiting-factor risk}_f}{D_{R,b}}
=
\sum_{s,l}\text{risk component}_{s,l,f,normalized}
$$

### Overall BSR summaries

The overall raw and normalized BSR scores are:

$$
\text{overall limiting-factor impact}
=
\sum_{s,l,f}\text{impact component}_{s,l,f}
$$

$$
\begin{aligned}
\text{overall limiting-factor impact}_{normalized}
&=
\frac{\text{overall limiting-factor impact}}{D_{I,b}}\\
&=
\sum_{s,l,f}\text{impact component}_{s,l,f,normalized}
\end{aligned}
$$

$$
\text{overall risk score}
=
\sum_{s,l,f}\text{risk component}_{s,l,f}
$$

$$
\begin{aligned}
\text{overall risk score}_{normalized}
&=
\frac{\text{overall risk score}}{D_{R,b}}\\
&=
\sum_{s,l,f}\text{risk component}_{s,l,f,normalized}
\end{aligned}
$$

Therefore:

$$
\sum_{s,l}\text{life-stage risk score}_{normalized}
=
\sum_f\text{limiting-factor risk}_{normalized}
=
\text{overall risk score}_{normalized}
$$

and the same identity holds for impact. Rankings are calculated within each BSR. Rank 1 identifies the largest contribution, and tied top contributors are retained together in the BSR summary.


In [ ]:
life_stage_scores = (
    calculation_grid.groupby(
        ["bsr", "basin", "species", "life_stage"], as_index=False
    )
    .agg(
        LS_corrected_score=("LS_corrected_score", "first"),
        species_aggregate_score=("species_aggregate_score", "first"),
        fish_use_score=("fish_use_score", "first"),
        population_priority=("population_priority", "first"),
        impact_score=("impact_component", "sum"),
        impact_score_normalized=("impact_component_normalized", "sum"),
        risk_score=("risk_component", "sum"),
        risk_score_normalized=("risk_component_normalized", "sum"),
        vulnerability_review_flag=("vulnerability_review_flag", any_yes),
    )
)
life_stage_scores["risk_rank_within_bsr"] = (
    life_stage_scores.groupby("bsr")["risk_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)
life_stage_scores["species_life_stage_label"] = (
    life_stage_scores["species"] + " | " + life_stage_scores["life_stage"]
)

limiting_factor_scores = (
    calculation_grid.groupby(
        ["bsr", "basin", "limiting_factor"], as_index=False
    )
    .agg(
        fish_use_score=("fish_use_score", "first"),
        condition_score_raw_1_5=("condition_score_raw_1_5", "first"),
        condition_score=("condition_score", "first"),
        condition_rating_n=("n", "first"),
        condition_review_flag=("any_flag", "first"),
        vulnerability_review_flag=("vulnerability_review_flag", any_yes),
        impact_score=("impact_component", "sum"),
        impact_score_normalized=("impact_component_normalized", "sum"),
        risk_score=("risk_component", "sum"),
        risk_score_normalized=("risk_component_normalized", "sum"),
    )
)
limiting_factor_scores["risk_rank_within_bsr"] = (
    limiting_factor_scores.groupby("bsr")["risk_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

source_fish_use = (
    fish_use.groupby(["bsr", "basin"], as_index=False)
    .agg(
        fish_use_score=("fish_use_score", "first"),
        fish_use_score_variants=("fish_use_score", "nunique"),
    )
)

bsr_from_life_stage = (
    life_stage_scores.groupby(["bsr", "basin"], as_index=False)
    .agg(
        overall_impact_score=("impact_score", "sum"),
        overall_impact_score_normalized=("impact_score_normalized", "sum"),
        overall_risk_score=("risk_score", "sum"),
        overall_risk_score_normalized=("risk_score_normalized", "sum"),
    )
)
bsr_from_limiting_factor = (
    limiting_factor_scores.groupby("bsr", as_index=False)
    .agg(
        lf_sum_impact_score=("impact_score", "sum"),
        lf_sum_impact_score_normalized=("impact_score_normalized", "sum"),
        lf_sum_risk_score=("risk_score", "sum"),
        lf_sum_risk_score_normalized=("risk_score_normalized", "sum"),
    )
)

top_life_stage = (
    life_stage_scores.loc[life_stage_scores["risk_rank_within_bsr"].eq(1)]
    .groupby("bsr", as_index=False)
    .agg(
        highest_risk_species_life_stage=(
            "species_life_stage_label",
            lambda values: "; ".join(sorted(values)),
        ),
        top_species_life_stage_risk_score=("risk_score", "first"),
        top_species_life_stage_risk_score_normalized=(
            "risk_score_normalized", "first"
        ),
        top_species_life_stage_risk_tie_count=(
            "species_life_stage_label", "size"
        ),
    )
)
top_limiting_factor = (
    limiting_factor_scores.loc[
        limiting_factor_scores["risk_rank_within_bsr"].eq(1)
    ]
    .groupby("bsr", as_index=False)
    .agg(
        highest_risk_limiting_factor=(
            "limiting_factor", lambda values: "; ".join(sorted(values))
        ),
        top_limiting_factor_risk_score=("risk_score", "first"),
        top_limiting_factor_risk_score_normalized=(
            "risk_score_normalized", "first"
        ),
        top_limiting_factor_risk_tie_count=(
            "limiting_factor", "size"
        ),
    )
)

bsr_scores = (
    bsr_from_life_stage
    .merge(bsr_from_limiting_factor, on="bsr", validate="one_to_one")
    .merge(source_fish_use, on=["bsr", "basin"], validate="one_to_one")
    .merge(top_life_stage, on="bsr", validate="one_to_one")
    .merge(top_limiting_factor, on="bsr", validate="one_to_one")
    .merge(
        normalization_denominators,
        on=["bsr", "basin"],
        validate="one_to_one",
    )
)
bsr_scores["impact_balance_difference"] = (
    bsr_scores["overall_impact_score"]
    - bsr_scores["lf_sum_impact_score"]
)
bsr_scores["risk_balance_difference"] = (
    bsr_scores["overall_risk_score"]
    - bsr_scores["lf_sum_risk_score"]
)
bsr_scores["impact_balance_difference_normalized"] = (
    bsr_scores["overall_impact_score_normalized"]
    - bsr_scores["lf_sum_impact_score_normalized"]
)
bsr_scores["risk_balance_difference_normalized"] = (
    bsr_scores["overall_risk_score_normalized"]
    - bsr_scores["lf_sum_risk_score_normalized"]
)
display(
    bsr_scores[
        [
            "bsr", "overall_risk_score", "overall_risk_score_normalized",
            "highest_risk_species_life_stage",
            "highest_risk_limiting_factor",
            "fish_use_score",
        ]
    ].sort_values("overall_risk_score", ascending=False).head(10)
)


## 6. Calculate Level 2 action-specific and overall scores

Level 2 applies the LFAT relationship between each action type and limiting factor to the Level 1 results. Original raw action fields are retained unchanged, with parallel `_normalized` fields added.

### Action-to-limiting-factor weight

The LFAT weight is supplied by the CSV and calculated for each action type and limiting factor as:

$$
\text{action weight}
=
\text{relationship directness}
\times \text{frequency}
$$

The corresponding notebook fields are `lfat_score`, `directness_value`, and `frequency_value`.

Three raw action-specific summaries are calculated for each BSR and action type:

$$
\text{\{action type\} condition improvement score}
=
\sum_f
\left(
\text{condition}_f
\times \text{action weight}_{a,f}
\right)
$$

$$
\text{\{action type\} limiting-factor amelioration score}
=
\sum_f
\left(
\text{limiting-factor impact}_f
\times \text{action weight}_{a,f}
\right)
$$

$$
\text{\{action type\} benefit score}
=
\sum_f
\left(
\text{limiting-factor risk}_f
\times \text{action weight}_{a,f}
\right)
$$

In the output table, the action-specific benefit field remains `action_benefit_score`. Its display label should use the action type, for example, `Floodplain Restoration benefit score`.

### Level 2 normalization

Let:

$$
W=\sum_a\sum_f \text{action weight}_{a,f}
$$

where $W$ is the sum of the complete LFAT crosswalk. For BSR $b$, the theoretical maxima are:

$$
D_C=W
$$

$$
D_{A,b}=J_b\times W
$$

$$
D_{B,b}=M_{LS}\times P_b\times W
$$

where $J_b$, $P_b$, and $M_{LS}$ are the Level 1 normalization terms defined above. The normalized action-specific scores are:

$$
\text{condition improvement score}_{a,normalized}
=
\frac{\text{condition improvement score}_a}{D_C}
$$

$$
\text{limiting-factor amelioration score}_{a,normalized}
=
\frac{\text{limiting-factor amelioration score}_a}{D_{A,b}}
$$

$$
\text{action benefit score}_{a,normalized}
=
\frac{\text{action benefit score}_a}{D_{B,b}}
$$

The denominators span **all action types**, not one action at a time. Therefore each normalized action score represents that action's contribution to the common 0-to-1 overall scale, rather than independently stretching every action to 1.

The raw overall BSR scores remain sums across all action types:

$$
\text{overall condition improvement score}
=
\sum_a \text{condition improvement score}_a
$$

$$
\text{overall limiting-factor amelioration score}
=
\sum_a \text{limiting-factor amelioration score}_a
$$

$$
\text{overall benefit score}
=
\sum_a \text{action benefit score}_a
$$

The normalized overall scores preserve the same accounting identities:

$$
\text{overall condition improvement score}_{normalized}
=
\sum_a \text{condition improvement score}_{a,normalized}
$$

$$
\text{overall limiting-factor amelioration score}_{normalized}
=
\sum_a \text{limiting-factor amelioration score}_{a,normalized}
$$

$$
\text{overall benefit score}_{normalized}
=
\sum_a \text{action benefit score}_{a,normalized}
$$

where:

- `condition_improvement_score` is the original action-specific score based only on condition and the action crosswalk;
- `limiting_factor_amelioration_score` is the original action-specific score that applies the action weight to limiting-factor impact;
- `action_benefit_score` is the original action-specific score that applies the action weight to limiting-factor risk;
- fields with the same base name plus `_normalized` place those scores on the shared 0-to-1 theoretical scale; and
- `overall_condition_improvement_score`, `overall_limiting_factor_amelioration_score`, and `overall_benefit_score` remain the original raw sums.

These scores represent relative alignment with the scored limiting factors. They do not estimate the realized benefit, cost, or feasibility of a specific project.


In [ ]:
action_components = limiting_factor_scores.merge(
    lfat[
        [
            "action_id", "action_type", "action_definition",
            "limiting_factor", "directness_code", "directness_value",
            "frequency_code", "frequency_value", "lfat_score",
        ]
    ],
    on="limiting_factor",
    how="inner",
    validate="many_to_many",
).merge(
    normalization_denominators[
        [
            "bsr", "basin",
            "condition_improvement_normalization_denominator",
            "amelioration_normalization_denominator",
            "benefit_normalization_denominator",
        ]
    ],
    on=["bsr", "basin"],
    how="left",
    validate="many_to_one",
)
action_components["condition_improvement_component"] = (
    action_components["condition_score"]
    * action_components["lfat_score"]
)
action_components["amelioration_component"] = (
    action_components["impact_score"]
    * action_components["lfat_score"]
)
action_components["benefit_component"] = (
    action_components["risk_score"]
    * action_components["lfat_score"]
)

# Normalize against the maximum possible total across the complete action
# crosswalk. The same denominator is used for every action in a BSR so the
# normalized action-specific scores still sum exactly to the normalized total.
action_components["condition_improvement_component_normalized"] = (
    action_components["condition_improvement_component"]
    / action_components["condition_improvement_normalization_denominator"]
)
action_components["amelioration_component_normalized"] = (
    action_components["amelioration_component"]
    / action_components["amelioration_normalization_denominator"]
)
action_components["benefit_component_normalized"] = (
    action_components["benefit_component"]
    / action_components["benefit_normalization_denominator"]
)

action_scores = (
    action_components.groupby(
        [
            "bsr", "basin", "action_id", "action_type",
            "action_definition",
        ],
        as_index=False,
    )
    .agg(
        condition_improvement_score=(
            "condition_improvement_component", "sum"
        ),
        condition_improvement_score_normalized=(
            "condition_improvement_component_normalized", "sum"
        ),
        limiting_factor_amelioration_score=(
            "amelioration_component", "sum"
        ),
        limiting_factor_amelioration_score_normalized=(
            "amelioration_component_normalized", "sum"
        ),
        action_benefit_score=("benefit_component", "sum"),
        action_benefit_score_normalized=(
            "benefit_component_normalized", "sum"
        ),
        condition_review_count=(
            "condition_review_flag",
            lambda values: int(
                values.fillna(False)
                .astype(str)
                .str.strip()
                .str.lower()
                .isin(["true", "yes", "1"])
                .sum()
            ),
        ),
        vulnerability_review_count=(
            "vulnerability_review_flag",
            lambda values: int(values.eq("Yes").sum()),
        ),
    )
)

# Use the action type in the display label so the score is identified as,
# for example, "Floodplain Restoration benefit score".
action_scores["benefit_score_label"] = (
    action_scores["action_type"].astype(str) + " benefit score"
)

action_scores["benefit_rank_within_bsr"] = (
    action_scores.groupby("bsr")["action_benefit_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

top_action = (
    action_scores.loc[action_scores["benefit_rank_within_bsr"].eq(1)]
    .assign(
        priority_action_label=lambda table: (
            table["action_id"].astype(str) + " | " + table["action_type"]
        )
    )
    .groupby("bsr", as_index=False)
    .agg(
        highest_risk_aligned_action_type=(
            "priority_action_label",
            lambda values: "; ".join(sorted(values)),
        ),
        highest_action_benefit_score=(
            "action_benefit_score", "first"
        ),
        highest_action_benefit_score_normalized=(
            "action_benefit_score_normalized", "first"
        ),
        top_action_benefit_tie_count=(
            "priority_action_label", "size"
        ),
    )
)

# Overall BSR scores are sums of the corresponding action-specific scores.
overall_action_scores = (
    action_scores.groupby("bsr", as_index=False)
    .agg(
        overall_condition_improvement_score=(
            "condition_improvement_score", "sum"
        ),
        overall_condition_improvement_score_normalized=(
            "condition_improvement_score_normalized", "sum"
        ),
        overall_limiting_factor_amelioration_score=(
            "limiting_factor_amelioration_score", "sum"
        ),
        overall_limiting_factor_amelioration_score_normalized=(
            "limiting_factor_amelioration_score_normalized", "sum"
        ),
        overall_benefit_score=("action_benefit_score", "sum"),
        overall_benefit_score_normalized=(
            "action_benefit_score_normalized", "sum"
        ),
    )
)

bsr_scores = (
    bsr_scores
    .merge(top_action, on="bsr", validate="one_to_one")
    .merge(overall_action_scores, on="bsr", validate="one_to_one")
)

display(
    action_scores[
        [
            "bsr", "action_type", "benefit_score_label",
            "condition_improvement_score",
            "condition_improvement_score_normalized",
            "limiting_factor_amelioration_score",
            "limiting_factor_amelioration_score_normalized",
            "action_benefit_score", "action_benefit_score_normalized",
            "benefit_rank_within_bsr",
        ]
    ].sort_values(["bsr", "benefit_rank_within_bsr"]).head(15)
)

display(
    bsr_scores[
        [
            "bsr", "overall_condition_improvement_score",
            "overall_condition_improvement_score_normalized",
            "overall_limiting_factor_amelioration_score",
            "overall_limiting_factor_amelioration_score_normalized",
            "overall_benefit_score", "overall_benefit_score_normalized",
        ]
    ].sort_values("overall_benefit_score", ascending=False).head(10)
)


## 7. Save core outputs, QC tables, and the scored GeoPackage

The seven core CSVs and scored GeoPackage are saved directly in `data/outputs`. The fish-use and population input tables are written both as CSVs and as nonspatial attribute tables in the GeoPackage. Supporting calculation components and review tables are saved in `data/outputs/QC`. The spatial output is required and is created on every successful run.


In [ ]:
identifier_review = (
    fish_use[["bsr", "basin", "bsr_crosswalk_status"]]
    .drop_duplicates()
    .sort_values(["basin", "bsr"])
    .reset_index(drop=True)
)

CORE_SCORE_FILES = {
    "bsr": "bsr_scores.csv",
    "fish_use": "fish_use_scores.csv",
    "population": "population_scores.csv",
    "life_stage": "life_stage_scores.csv",
    "limiting_factor": "limiting_factor_scores_integrated.csv",
    "action": "action_scores.csv",
    "grid": "calculation_grid.csv",
}

CORE_OUTPUTS = {
    CORE_SCORE_FILES["bsr"]: bsr_scores,
    CORE_SCORE_FILES["fish_use"]: fish_use,
    CORE_SCORE_FILES["population"]: population,
    CORE_SCORE_FILES["life_stage"]: life_stage_scores,
    CORE_SCORE_FILES["limiting_factor"]: limiting_factor_scores,
    CORE_SCORE_FILES["action"]: action_scores,
    CORE_SCORE_FILES["grid"]: calculation_grid,
}
QC_OUTPUTS = {
    "action_score_components.csv": action_components,
    "assumptions_for_review.csv": assumptions,
    "bsr_identifiers_for_review.csv": identifier_review,
    "normalization_denominators.csv": normalization_denominators,
    "vulnerability_scores_for_review.csv": vulnerability_collapsed,
}

for filename, table in CORE_OUTPUTS.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)
for filename, table in QC_OUTPUTS.items():
    table.to_csv(QC_DIR / filename, index=False)

ALL_CSV_OUTPUTS = {**CORE_OUTPUTS, **QC_OUTPUTS}
OUTPUT_PATHS = {
    **{name: OUTPUT_DIR / name for name in CORE_OUTPUTS},
    **{name: QC_DIR / name for name in QC_OUTPUTS},
}
GPKG_ATTRIBUTE_TABLES = {
    "fish_use_scores": {
        "table": fish_use,
        "identifier": "Atlas fish-use scores",
        "description": (
            "Source overall, species, and life-stage fish-use scores"
        ),
    },
    "population_scores": {
        "table": population,
        "identifier": "Atlas population scores",
        "description": (
            "Source population-priority scores by basin, species, and life stage"
        ),
    },
    "life_stage_scores": {
        "table": life_stage_scores,
        "identifier": "Atlas life-stage scores",
        "description": (
            "Level 1 species and life-stage scores by BSR for attribute joins"
        ),
    },
    "limiting_factor_scores": {
        "table": limiting_factor_scores,
        "identifier": "Atlas limiting-factor scores",
        "description": (
            "Level 1 limiting-factor scores by BSR for attribute joins"
        ),
    },
    "action_type_scores": {
        "table": action_scores,
        "identifier": "Atlas action-type scores",
        "description": (
            "Level 2 action-type scores by BSR for attribute joins"
        ),
    },
}


def quote_identifier(name):
    return '"' + str(name).replace('"', '""') + '"'


def sqlite_type(series):
    if pd.api.types.is_bool_dtype(series) or pd.api.types.is_integer_dtype(series):
        return "INTEGER"
    if pd.api.types.is_numeric_dtype(series):
        return "REAL"
    return "TEXT"


def sqlite_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


def geometry_digest(path, feature_table, key_field, geometry_field):
    digest = hashlib.sha256()
    query = (
        f"SELECT {quote_identifier(key_field)}, "
        f"{quote_identifier(geometry_field)} "
        f"FROM {quote_identifier(feature_table)} "
        f"ORDER BY {quote_identifier(key_field)}"
    )
    with closing(sqlite3.connect(path)) as connection:
        for key, geometry in connection.execute(query):
            digest.update(str(key).encode("utf-8"))
            digest.update(bytes(geometry) if geometry is not None else b"")
    return digest.hexdigest()


def write_attribute_table(
    connection, table_name, table, identifier, description
):
    columns = table.columns.tolist()
    normalized_columns = [str(column).lower() for column in columns]
    if len(normalized_columns) != len(set(normalized_columns)):
        raise ValueError(
            f"GeoPackage table {table_name} has duplicate column names."
        )
    if "fid" in normalized_columns:
        raise ValueError(
            f"GeoPackage table {table_name} already contains a fid field."
        )
    table_exists = connection.execute(
        "SELECT 1 FROM sqlite_master WHERE name = ?", (table_name,)
    ).fetchone()
    if table_exists is not None:
        raise ValueError(
            f"GeoPackage already contains a table named {table_name}."
        )

    column_definitions = ["fid INTEGER PRIMARY KEY AUTOINCREMENT"]
    column_definitions.extend(
        f"{quote_identifier(column)} {sqlite_type(table[column])}"
        for column in columns
    )
    connection.execute(
        f"CREATE TABLE {quote_identifier(table_name)} "
        f"({', '.join(column_definitions)})"
    )

    placeholders = ", ".join("?" for _ in columns)
    insert_sql = (
        f"INSERT INTO {quote_identifier(table_name)} "
        f"({', '.join(quote_identifier(column) for column in columns)}) "
        f"VALUES ({placeholders})"
    )
    connection.executemany(
        insert_sql,
        (
            tuple(sqlite_value(value) for value in row)
            for row in table.itertuples(index=False, name=None)
        ),
    )
    connection.execute(
        "INSERT INTO gpkg_contents "
        "(table_name, data_type, identifier, description, last_change) "
        "VALUES (?, 'attributes', ?, ?, "
        "strftime('%Y-%m-%dT%H:%M:%fZ', 'now'))",
        (table_name, identifier, description),
    )
    if "bsr" in normalized_columns:
        bsr_column = columns[normalized_columns.index("bsr")]
        index_name = f"idx_{table_name}_bsr"
        connection.execute(
            f"CREATE INDEX {quote_identifier(index_name)} "
            f"ON {quote_identifier(table_name)} "
            f"({quote_identifier(bsr_column)})"
        )


def write_scored_bsr_gpkg(
    input_path, output_path, summary, attribute_tables
):
    # sqlite3 connections are closed explicitly because its context
    # manager commits or rolls back but does not close the file handle.
    # An open handle prevents Path.replace() on Windows.
    with closing(sqlite3.connect(input_path)) as connection:
        feature_tables = [
            row[0]
            for row in connection.execute(
                "SELECT table_name FROM gpkg_contents WHERE data_type = 'features'"
            )
        ]
        candidates = []
        for table_name in feature_tables:
            columns = [
                row[1]
                for row in connection.execute(
                    f"PRAGMA table_info({quote_identifier(table_name)})"
                )
            ]
            source_key = next(
                (column for column in columns if column.lower() == "bsr"),
                None,
            )
            if source_key is not None:
                candidates.append((table_name, source_key, columns))

        if len(candidates) != 1:
            raise ValueError(
                "Expected exactly one feature layer containing a BSR field; "
                f"found {len(candidates)}."
            )
        feature_table, source_key, source_columns = candidates[0]
        geometry_row = connection.execute(
            "SELECT column_name FROM gpkg_geometry_columns WHERE table_name = ?",
            (feature_table,),
        ).fetchone()
        if geometry_row is None:
            raise ValueError(
                f"No geometry field is registered for layer {feature_table}."
            )
        geometry_field = geometry_row[0]
        spatial_keys = {
            str(row[0]).strip()
            for row in connection.execute(
                f"SELECT {quote_identifier(source_key)} "
                f"FROM {quote_identifier(feature_table)}"
            )
        }

    summary_keys = set(summary["bsr"].astype(str).str.strip())
    if spatial_keys != summary_keys:
        raise ValueError(
            "The GeoPackage and score summary have different BSR coverage. "
            f"Missing scores: {sorted(spatial_keys - summary_keys)}; "
            f"missing geometry: {sorted(summary_keys - spatial_keys)}"
        )

    scored = summary.rename(columns={"bsr": "score_bsr"}).copy()
    scored_columns = scored.columns.tolist()
    existing_lower = {column.lower() for column in source_columns}
    collisions = [
        column for column in scored_columns if column.lower() in existing_lower
    ]
    if collisions:
        raise ValueError(
            "Score fields collide with existing GeoPackage fields: "
            + ", ".join(collisions)
        )

    temporary_path = output_path.with_name(
        f".{output_path.stem}.{uuid4().hex}.tmp.gpkg"
    )
    shutil.copy2(input_path, temporary_path)

    try:
        with closing(sqlite3.connect(temporary_path)) as connection:
            # Some GeoPackages contain RTree update triggers that invoke these
            # functions even when only non-geometry fields are updated.
            connection.create_function(
                "ST_IsEmpty", 1, lambda geometry: 1 if geometry is None else 0
            )
            for function_name in ("ST_MinX", "ST_MaxX", "ST_MinY", "ST_MaxY"):
                connection.create_function(
                    function_name, 1, lambda geometry: 0.0
                )

            with connection:
                for column in scored_columns:
                    connection.execute(
                        f"ALTER TABLE {quote_identifier(feature_table)} "
                        f"ADD COLUMN {quote_identifier(column)} "
                        f"{sqlite_type(scored[column])}"
                    )

                assignments = ", ".join(
                    f"{quote_identifier(column)} = ?"
                    for column in scored_columns
                )
                update_sql = (
                    f"UPDATE {quote_identifier(feature_table)} "
                    f"SET {assignments} "
                    f"WHERE TRIM(CAST("
                    f"{quote_identifier(source_key)} AS TEXT)) = ?"
                )
                for original_bsr, (_, row) in zip(
                    summary["bsr"].astype(str), scored.iterrows()
                ):
                    values = [
                        sqlite_value(row[column])
                        for column in scored_columns
                    ]
                    values.append(original_bsr.strip())
                    cursor = connection.execute(update_sql, values)
                    if cursor.rowcount != 1:
                        raise ValueError(
                            f"Expected one spatial row for {original_bsr}; "
                            f"updated {cursor.rowcount}."
                        )

                connection.execute(
                    "UPDATE gpkg_contents "
                    "SET identifier = ?, description = ? "
                    "WHERE table_name = ?",
                    (
                        "Atlas scored BSRs",
                        "BSR geometry with Level 1 and Level 2 score summaries",
                        feature_table,
                    ),
                )

                for table_name, table_specification in attribute_tables.items():
                    write_attribute_table(
                        connection=connection,
                        table_name=table_name,
                        **table_specification,
                    )

            integrity = connection.execute(
                "PRAGMA integrity_check"
            ).fetchone()[0]
            if integrity != "ok":
                raise ValueError(
                    f"GeoPackage integrity check failed: {integrity}"
                )

        # Path.replace() overwrites an existing output atomically. The SQLite
        # connection must be closed before this line on Windows.
        try:
            temporary_path.replace(output_path)
        except PermissionError as error:
            raise PermissionError(
                f"Could not replace {output_path}. Close the output "
                "GeoPackage in QGIS, ArcGIS, or another application, then "
                "run this cell again."
            ) from error
    finally:
        # Do not let cleanup of a staging file hide the original exception.
        try:
            temporary_path.unlink(missing_ok=True)
        except OSError:
            pass

    return (
        feature_table, geometry_field, source_key, scored_columns,
        list(attribute_tables),
    )


(
    BSR_OUTPUT_LAYER,
    BSR_GEOMETRY_FIELD,
    BSR_SOURCE_KEY,
    BSR_SCORE_FIELDS,
    BSR_ATTRIBUTE_TABLES,
) = write_scored_bsr_gpkg(
    BSR_INPUT_PATH, BSR_OUTPUT_PATH, bsr_scores, GPKG_ATTRIBUTE_TABLES
)

manifest_rows = [
    {"file": filename, "rows": len(table), "path": OUTPUT_PATHS[filename]}
    for filename, table in ALL_CSV_OUTPUTS.items()
]
manifest_rows.append(
    {
        "file": BSR_OUTPUT_FILE,
        "rows": len(bsr_scores),
        "path": BSR_OUTPUT_PATH,
    }
)
output_manifest = pd.DataFrame(manifest_rows)
display(output_manifest)


## 8. Interpretations and notes

- All original raw score field names are retained. Parallel fields ending in `_normalized` place impact, risk, condition improvement, limiting-factor amelioration, and benefit on a theoretical 0-to-1 scale.
- Normalized scores are **not** percentiles and are not min-max scaled to the current BSR dataset. A normalized overall score of 0.60 means 60% of the maximum possible score under the fixed scoring structure and supplied weighting framework.
- Normalized life-stage, limiting-factor, and action-specific scores are contributions to their common overall 0-to-1 total. They therefore may have maxima well below 1 individually, and they remain exactly additive to the normalized overall score.
- Raw aggregate scores remain sums across multiple species, life stages, limiting factors, or action types and can exceed 1. They are retained for full auditability and backward compatibility.
- BSRs are directly comparable here because the quality-control checks require the same species × life-stage, limiting-factor, and action-type coverage for every BSR. The risk and benefit normalization denominators also use the supplied population-priority totals, preserving any small source rounding differences rather than silently changing the raw weights.
- `action_benefit_score` remains the original action-specific score. `action_benefit_score_normalized` is that action's contribution to the common 0-to-1 overall benefit scale. The display label can continue to use the action type, such as `Floodplain Restoration benefit score`.
- `overall_condition_improvement_score`, `overall_limiting_factor_amelioration_score`, and `overall_benefit_score` remain the original raw sums. Their `_normalized` counterparts are the corresponding 0-to-1 theoretical-scale sums.
- A high action benefit score indicates that an action type aligns with limiting factors contributing to calculated risk. A high `overall_benefit_score_normalized` indicates a larger fraction of the maximum possible summed alignment across the full LFAT crosswalk.
- Summing across action types can count the same limiting-factor pathway more than once. Normalization does not remove that structural overlap; it only places the aggregate alignment on a common theoretical scale.
- None of the action scores account for project feasibility, cost, landowner willingness, implementation constraints, or site-specific effectiveness.
- The maximum adult/juvenile vulnerability used for combined migration is a conservative aggregation choice. The detailed review table retains the source stages and score range.


## 9. Quality control

The checks below verify the attached schemas, score transformations, key uniqueness, coverage of all 15 limiting factors, completeness of every join, reconciliation of alternative aggregation paths, saved outputs, GeoPackage integrity, and geometry preservation. An assertion stops the notebook at the first failed requirement rather than allowing incomplete scores to propagate.


In [ ]:
# QC 1: source values and transformations
required_non_null = {
    "fish-use BSR": fish_use["bsr"],
    "life-stage fish-use score": fish_use["LS_corrected_score"],
    "species fish-use score": fish_use["species_aggregate_score"],
    "BSR fish-use score": fish_use["fish_use_score"],
    "population priority": population["population_priority"],
    "condition raw score": condition["condition_score_raw_1_5"],
    "condition score": condition["condition_score"],
    "vulnerability rank": vulnerability["vulnerability_rank"],
    "vulnerability score": vulnerability["vulnerability_score"],
    "LFAT directness": lfat["directness_value"],
    "LFAT frequency": lfat["frequency_value"],
    "LFAT score": lfat["lfat_score"],
}
unresolved = {
    name: int(series.isna().sum())
    for name, series in required_non_null.items()
}
assert not any(unresolved.values()), unresolved

assert fish_use["fish_use_score"].between(0, 1).all()
assert fish_use["LS_corrected_score"].between(0, LIFE_STAGE_FISH_USE_MAX).all()
assert fish_use["species_aggregate_score"].ge(0).all()
assert population["population_priority"].between(0, 1).all()

population_sums = (
    population.groupby(["basin", "species"])["population_priority"].sum()
)
assert np.allclose(population_sums, 1.0, atol=0.011)

expected_condition_score = 0.10 + (
    condition["condition_score_raw_1_5"] - 1.0
) * (0.90 / 4.0)
assert condition["condition_score_raw_1_5"].between(1, 5).all()
assert condition["condition_score"].between(0.10, 1.00).all()
assert np.allclose(
    condition["condition_score"], expected_condition_score, atol=1e-12
)

expected_vulnerability_score = 1.0 - 0.9 * (
    vulnerability["vulnerability_rank"] - 1.0
) / 14.0
assert vulnerability["vulnerability_rank"].between(1, 15).all()
assert vulnerability["vulnerability_score"].between(0.10, 1.00).all()
assert np.allclose(
    vulnerability["vulnerability_score"],
    expected_vulnerability_score,
    atol=1e-8,
)

vulnerability_source_coverage = vulnerability.groupby(
    ["species", "source_life_stage"]
).agg(
    rows=("limiting_factor", "size"),
    limiting_factors=("limiting_factor", "nunique"),
)
# Tied source ranks are valid. Every source stage must still contain one
# record for each of the 15 limiting factors.
assert vulnerability_source_coverage["rows"].eq(15).all()
assert vulnerability_source_coverage["limiting_factors"].eq(15).all()

assert np.allclose(
    lfat["lfat_score"],
    lfat["directness_value"] * lfat["frequency_value"],
    atol=1e-12,
)
assert lfat["lfat_score"].between(0, 1).all()

assert LFAT_TOTAL_WEIGHT > 0
normalization_fields = [
    "impact_normalization_denominator",
    "risk_normalization_denominator",
    "condition_improvement_normalization_denominator",
    "amelioration_normalization_denominator",
    "benefit_normalization_denominator",
]
assert normalization_denominators[normalization_fields].gt(0).all().all()

display(population_sums.rename("priority_sum").reset_index())
display(pd.DataFrame([unresolved]).T.rename(columns={0: "missing_values"}))


In [ ]:
# QC 2: keys, coverage, joins, balances, and saved outputs
assert not fish_use.duplicated(["bsr", "species", "life_stage"]).any()
assert not population.duplicated(["basin", "species", "life_stage"]).any()
assert not condition.duplicated(["bsr", "limiting_factor"]).any()
assert not vulnerability.duplicated(
    ["species", "source_life_stage", "limiting_factor"]
).any()
assert not lfat.duplicated(["action_id", "limiting_factor"]).any()
assert not vulnerability_collapsed.duplicated(
    ["species", "life_stage", "limiting_factor"]
).any()

canonical_lf_set = set(CANONICAL_LF)
assert set(condition["limiting_factor"]) == canonical_lf_set
assert set(vulnerability["limiting_factor"]) == canonical_lf_set
assert set(lfat["limiting_factor"]) == canonical_lf_set
assert set(fish_use["bsr"]) == set(condition["bsr"])
assert condition.groupby("bsr")["limiting_factor"].nunique().eq(15).all()
assert lfat.groupby("action_id")["limiting_factor"].nunique().eq(15).all()

fish_population_keys = set(
    map(
        tuple,
        fish_use[["basin", "species", "life_stage"]]
        .drop_duplicates()
        .to_numpy(),
    )
)
population_keys = set(
    map(tuple, population[["basin", "species", "life_stage"]].to_numpy())
)
assert fish_population_keys == population_keys

fish_stage_keys = set(
    map(
        tuple,
        fish_use[["species", "life_stage"]].drop_duplicates().to_numpy(),
    )
)
vulnerability_stage_keys = set(
    map(
        tuple,
        vulnerability_collapsed[["species", "life_stage"]]
        .drop_duplicates()
        .to_numpy(),
    )
)
assert fish_stage_keys == vulnerability_stage_keys

expected_vulnerability_rows = len(fish_stage_keys) * len(CANONICAL_LF)
expected_grid_rows = len(fish_use) * len(CANONICAL_LF)
required_grid_fields = [
    "LS_corrected_score", "species_aggregate_score",
    "fish_use_score", "population_priority", "vulnerability_score",
    "condition_score", "impact_component", "risk_component",
    "impact_component_normalized", "risk_component_normalized",
    "impact_normalization_denominator",
    "risk_normalization_denominator",
]
assert len(vulnerability_collapsed) == expected_vulnerability_rows
assert len(calculation_grid) == expected_grid_rows
assert not calculation_grid[required_grid_fields].isna().any().any()
assert calculation_grid.groupby("bsr").size().eq(
    len(fish_stage_keys) * len(CANONICAL_LF)
).all()
assert source_fish_use["fish_use_score_variants"].eq(1).all()
assert fish_use.groupby(["bsr", "species"])[
    "species_aggregate_score"
].nunique().eq(1).all()
score_keys = ["bsr", "basin", "species", "life_stage"]
source_detail_scores = fish_use.set_index(score_keys)[
    ["LS_corrected_score", "species_aggregate_score"]
].sort_index()
output_detail_scores = life_stage_scores.set_index(score_keys)[
    ["LS_corrected_score", "species_aggregate_score"]
].sort_index()
pd.testing.assert_frame_equal(output_detail_scores, source_detail_scores)
assert np.allclose(
    calculation_grid["impact_component"],
    calculation_grid["fish_use_score"]
    * calculation_grid["condition_score"]
    * calculation_grid["vulnerability_score"],
)
assert np.allclose(
    calculation_grid["risk_component"],
    calculation_grid["LS_corrected_score"]
    * calculation_grid["condition_score"]
    * calculation_grid["vulnerability_score"]
    * calculation_grid["population_priority"],
)
assert np.allclose(
    calculation_grid["impact_component_normalized"],
    calculation_grid["impact_component"]
    / calculation_grid["impact_normalization_denominator"],
)
assert np.allclose(
    calculation_grid["risk_component_normalized"],
    calculation_grid["risk_component"]
    / calculation_grid["risk_normalization_denominator"],
)
assert np.allclose(
    calculation_grid.loc[
        calculation_grid["LS_corrected_score"].eq(0), "risk_component"
    ],
    0.0,
)
assert np.allclose(
    life_stage_scores.loc[
        life_stage_scores["species"].eq("Bull Trout")
        & life_stage_scores["species_aggregate_score"].eq(0),
        "risk_score",
    ],
    0.0,
)
top_life_stage_rows = life_stage_scores.loc[
    life_stage_scores["risk_rank_within_bsr"].eq(1)
]
positive_risk_bsrs = set(
    life_stage_scores.groupby("bsr")["risk_score"]
    .max()
    .loc[lambda values: values > 0]
    .index
)
assert top_life_stage_rows.loc[
    top_life_stage_rows["bsr"].isin(positive_risk_bsrs),
    "LS_corrected_score",
].gt(0).all()
assert np.allclose(bsr_scores["impact_balance_difference"], 0.0)
assert np.allclose(bsr_scores["risk_balance_difference"], 0.0)
assert np.allclose(
    bsr_scores["impact_balance_difference_normalized"], 0.0
)
assert np.allclose(
    bsr_scores["risk_balance_difference_normalized"], 0.0
)
assert np.allclose(
    bsr_scores["overall_impact_score_normalized"],
    bsr_scores["overall_impact_score"]
    / bsr_scores["impact_normalization_denominator"],
)
assert np.allclose(
    bsr_scores["overall_risk_score_normalized"],
    bsr_scores["overall_risk_score"]
    / bsr_scores["risk_normalization_denominator"],
)

# All normalized fields use theoretical maxima and must remain within 0 to 1.
def assert_unit_interval(series, tolerance=1e-12):
    assert series.ge(-tolerance).all()
    assert series.le(1.0 + tolerance).all()

for table, columns in [
    (calculation_grid, [
        "impact_component_normalized", "risk_component_normalized",
    ]),
    (life_stage_scores, [
        "impact_score_normalized", "risk_score_normalized",
    ]),
    (limiting_factor_scores, [
        "impact_score_normalized", "risk_score_normalized",
    ]),
    (action_scores, [
        "condition_improvement_score_normalized",
        "limiting_factor_amelioration_score_normalized",
        "action_benefit_score_normalized",
    ]),
    (bsr_scores, [
        "overall_impact_score_normalized",
        "overall_risk_score_normalized",
        "overall_condition_improvement_score_normalized",
        "overall_limiting_factor_amelioration_score_normalized",
        "overall_benefit_score_normalized",
    ]),
]:
    for column in columns:
        assert_unit_interval(table[column])

# Confirm that each overall BSR action score is exactly the sum of its
# corresponding action-specific scores.
action_score_sums = (
    action_scores.groupby("bsr")
    .agg(
        overall_condition_improvement_score=(
            "condition_improvement_score", "sum"
        ),
        overall_condition_improvement_score_normalized=(
            "condition_improvement_score_normalized", "sum"
        ),
        overall_limiting_factor_amelioration_score=(
            "limiting_factor_amelioration_score", "sum"
        ),
        overall_limiting_factor_amelioration_score_normalized=(
            "limiting_factor_amelioration_score_normalized", "sum"
        ),
        overall_benefit_score=("action_benefit_score", "sum"),
        overall_benefit_score_normalized=(
            "action_benefit_score_normalized", "sum"
        ),
    )
    .sort_index()
)
bsr_action_totals = (
    bsr_scores.set_index("bsr")[
        [
            "overall_condition_improvement_score",
            "overall_condition_improvement_score_normalized",
            "overall_limiting_factor_amelioration_score",
            "overall_limiting_factor_amelioration_score_normalized",
            "overall_benefit_score",
            "overall_benefit_score_normalized",
        ]
    ]
    .sort_index()
)
pd.testing.assert_frame_equal(
    bsr_action_totals,
    action_score_sums,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)
assert all(path.exists() for path in OUTPUT_PATHS.values())
for filename, source_table in {
    CORE_SCORE_FILES["fish_use"]: fish_use,
    CORE_SCORE_FILES["population"]: population,
}.items():
    pd.testing.assert_frame_equal(
        pd.read_csv(OUTPUT_PATHS[filename]),
        source_table,
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12,
    )
expected_gpkg_attribute_rows = {
    name: len(specification["table"])
    for name, specification in GPKG_ATTRIBUTE_TABLES.items()
}

qc_rows = [
    ["Fish-use BSRs", fish_use["bsr"].nunique(), condition["bsr"].nunique()],
    ["Condition BSRs", condition["bsr"].nunique(), fish_use["bsr"].nunique()],
    ["Canonical limiting factors", len(CANONICAL_LF), 15],
    ["Collapsed vulnerability relationships", len(vulnerability_collapsed), expected_vulnerability_rows],
    ["Level 1 calculation-grid rows", len(calculation_grid), expected_grid_rows],
    ["LFAT action-factor relationships", len(lfat), lfat["action_id"].nunique() * len(CANONICAL_LF)],
    ["Saved CSV tables", sum(path.exists() for path in OUTPUT_PATHS.values()), len(OUTPUT_PATHS)],
]

assert BSR_OUTPUT_PATH.exists()
with closing(sqlite3.connect(BSR_OUTPUT_PATH)) as connection:
    scored_row_count = connection.execute(
        f"SELECT COUNT(*) FROM {quote_identifier(BSR_OUTPUT_LAYER)}"
    ).fetchone()[0]
    scored_null_count = connection.execute(
        f"SELECT COUNT(*) FROM {quote_identifier(BSR_OUTPUT_LAYER)} "
        "WHERE score_bsr IS NULL OR overall_risk_score IS NULL "
        f"OR {quote_identifier(BSR_GEOMETRY_FIELD)} IS NULL"
    ).fetchone()[0]
    gpkg_attribute_rows = {
        table_name: connection.execute(
            f"SELECT COUNT(*) FROM {quote_identifier(table_name)}"
        ).fetchone()[0]
        for table_name in BSR_ATTRIBUTE_TABLES
    }
    registered_attribute_tables = {
        row[0]
        for row in connection.execute(
            "SELECT table_name FROM gpkg_contents "
            "WHERE data_type = 'attributes'"
        )
    }
    scored_integrity = connection.execute(
        "PRAGMA integrity_check"
    ).fetchone()[0]

assert scored_row_count == len(bsr_scores)
assert scored_null_count == 0
assert gpkg_attribute_rows == expected_gpkg_attribute_rows
assert set(BSR_ATTRIBUTE_TABLES).issubset(registered_attribute_tables)
assert scored_integrity == "ok"
assert geometry_digest(
    BSR_INPUT_PATH,
    BSR_OUTPUT_LAYER,
    BSR_SOURCE_KEY,
    BSR_GEOMETRY_FIELD,
) == geometry_digest(
    BSR_OUTPUT_PATH,
    BSR_OUTPUT_LAYER,
    BSR_SOURCE_KEY,
    BSR_GEOMETRY_FIELD,
)
qc_rows.extend(
    [
        ["Scored GeoPackage BSR rows", scored_row_count, len(bsr_scores)],
        ["Scored GeoPackage null score/geometry rows", scored_null_count, 0],
        *[
            [
                f"GeoPackage {table_name} rows",
                gpkg_attribute_rows[table_name],
                expected_gpkg_attribute_rows[table_name],
            ]
            for table_name in BSR_ATTRIBUTE_TABLES
        ],
    ]
)

qc_summary = pd.DataFrame(
    qc_rows, columns=["check", "observed", "expected"]
)
qc_summary["pass"] = qc_summary["observed"].eq(qc_summary["expected"])
assert qc_summary["pass"].all()

display(qc_summary)
display(
    identifier_review.query("bsr_crosswalk_status != 'exact_identifier'")
)


### QC 3. Hand-calculated example

This small example uses one hypothetical BSR-level fish-use score, separate life-stage fish-use scores, two limiting factors, and two actions. It verifies that overall impact uses the BSR-level score, population-weighted risk uses the applicable life-stage score, and a zero life-stage score produces zero risk. It also verifies the normalization equations and confirms that normalized child scores still sum exactly to normalized overall scores. The example values test the equations only; they do not replace or recalibrate the input CSV scores.


In [ ]:
test_population = {"Spawning": 0.60, "Rearing": 0.40}
test_fish_use_score = 0.75
test_life_stage_fish_use = {"Spawning": 0.00, "Rearing": 0.80}
test_condition = {"Temperature": 0.10, "Instream Complexity": 1.00}
test_vulnerability = {
    ("Spawning", "Temperature"): 1.00,
    ("Rearing", "Temperature"): 0.50,
    ("Spawning", "Instream Complexity"): 0.50,
    ("Rearing", "Instream Complexity"): 1.00,
}
test_action_weight = {
    ("Floodplain Restoration", "Temperature"): 0.50,
    ("Floodplain Restoration", "Instream Complexity"): 1.00,
    ("Riparian Planting", "Temperature"): 1.00,
    ("Riparian Planting", "Instream Complexity"): 0.50,
}

# Theoretical maxima for this reduced example use the same normalization
# logic as the full model.
test_n_life_stages = len(test_population)
test_n_limiting_factors = len(test_condition)
test_population_priority_total = sum(test_population.values())
test_lfat_total_weight = sum(test_action_weight.values())
test_impact_denominator = (
    test_n_life_stages * test_n_limiting_factors
)
test_risk_denominator = (
    LIFE_STAGE_FISH_USE_MAX
    * test_population_priority_total
    * test_n_limiting_factors
)
test_condition_improvement_denominator = test_lfat_total_weight
test_amelioration_denominator = (
    test_n_life_stages * test_lfat_total_weight
)
test_benefit_denominator = (
    LIFE_STAGE_FISH_USE_MAX
    * test_population_priority_total
    * test_lfat_total_weight
)

test_rows = []
for life_stage in test_population:
    for limiting_factor, condition_score in test_condition.items():
        impact = (
            test_fish_use_score
            * condition_score
            * test_vulnerability[(life_stage, limiting_factor)]
        )
        risk = (
            test_life_stage_fish_use[life_stage]
            * condition_score
            * test_vulnerability[(life_stage, limiting_factor)]
            * test_population[life_stage]
        )
        test_rows.append(
            {
                "life_stage": life_stage,
                "limiting_factor": limiting_factor,
                "fish_use_score": test_fish_use_score,
                "LS_corrected_score": test_life_stage_fish_use[life_stage],
                "condition_score": condition_score,
                "impact_component": impact,
                "risk_component": risk,
                "impact_component_normalized": (
                    impact / test_impact_denominator
                ),
                "risk_component_normalized": (
                    risk / test_risk_denominator
                ),
            }
        )
test_grid = pd.DataFrame(test_rows)

test_life = test_grid.groupby("life_stage", as_index=False).agg(
    impact_score=("impact_component", "sum"),
    impact_score_normalized=("impact_component_normalized", "sum"),
    risk_score=("risk_component", "sum"),
    risk_score_normalized=("risk_component_normalized", "sum"),
)
test_lf = test_grid.groupby("limiting_factor", as_index=False).agg(
    condition_score=("condition_score", "first"),
    impact_score=("impact_component", "sum"),
    impact_score_normalized=("impact_component_normalized", "sum"),
    risk_score=("risk_component", "sum"),
    risk_score_normalized=("risk_component_normalized", "sum"),
)

test_actions = []
for action in ["Floodplain Restoration", "Riparian Planting"]:
    rows = test_lf.copy()
    rows["weight"] = rows["limiting_factor"].map(
        lambda factor: test_action_weight[(action, factor)]
    )
    condition_score = (
        rows["condition_score"] * rows["weight"]
    ).sum()
    amelioration_score = (
        rows["impact_score"] * rows["weight"]
    ).sum()
    benefit_score = (
        rows["risk_score"] * rows["weight"]
    ).sum()
    test_actions.append(
        {
            "action": action,
            "benefit_score_label": f"{action} benefit score",
            "condition_improvement_score": condition_score,
            "condition_improvement_score_normalized": (
                condition_score / test_condition_improvement_denominator
            ),
            "limiting_factor_amelioration_score": amelioration_score,
            "limiting_factor_amelioration_score_normalized": (
                amelioration_score / test_amelioration_denominator
            ),
            "action_benefit_score": benefit_score,
            "action_benefit_score_normalized": (
                benefit_score / test_benefit_denominator
            ),
        }
    )
test_actions = pd.DataFrame(test_actions)

test_overall_action_scores = pd.Series(
    {
        "overall_condition_improvement_score": (
            test_actions["condition_improvement_score"].sum()
        ),
        "overall_condition_improvement_score_normalized": (
            test_actions["condition_improvement_score_normalized"].sum()
        ),
        "overall_limiting_factor_amelioration_score": (
            test_actions["limiting_factor_amelioration_score"].sum()
        ),
        "overall_limiting_factor_amelioration_score_normalized": (
            test_actions[
                "limiting_factor_amelioration_score_normalized"
            ].sum()
        ),
        "overall_benefit_score": test_actions["action_benefit_score"].sum(),
        "overall_benefit_score_normalized": (
            test_actions["action_benefit_score_normalized"].sum()
        ),
    }
)

expected_life = {
    "Spawning": (0.45, 0.00),
    "Rearing": (0.7875, 0.336),
}
for row in test_life.itertuples(index=False):
    expected_impact, expected_risk = expected_life[row.life_stage]
    assert np.isclose(row.impact_score, expected_impact)
    assert np.isclose(row.risk_score, expected_risk)
    assert np.isclose(
        row.impact_score_normalized,
        expected_impact / test_impact_denominator,
    )
    assert np.isclose(
        row.risk_score_normalized,
        expected_risk / test_risk_denominator,
    )

assert test_grid["fish_use_score"].eq(test_fish_use_score).all()
assert test_grid.loc[
    test_grid["LS_corrected_score"].eq(0), "risk_component"
].eq(0).all()
assert np.isclose(test_lf["impact_score"].sum(), 1.2375)
assert np.isclose(test_lf["risk_score"].sum(), 0.336)
assert np.isclose(
    test_lf["impact_score_normalized"].sum(),
    1.2375 / test_impact_denominator,
)
assert np.isclose(
    test_lf["risk_score_normalized"].sum(),
    0.336 / test_risk_denominator,
)
assert np.allclose(
    test_actions["condition_improvement_score"], [1.05, 0.60]
)
assert np.allclose(
    test_actions["limiting_factor_amelioration_score"], [1.18125, 0.675]
)
assert np.allclose(test_actions["action_benefit_score"], [0.328, 0.176])
assert np.allclose(
    test_actions["condition_improvement_score_normalized"],
    np.array([1.05, 0.60]) / test_condition_improvement_denominator,
)
assert np.allclose(
    test_actions["limiting_factor_amelioration_score_normalized"],
    np.array([1.18125, 0.675]) / test_amelioration_denominator,
)
assert np.allclose(
    test_actions["action_benefit_score_normalized"],
    np.array([0.328, 0.176]) / test_benefit_denominator,
)
assert np.isclose(
    test_overall_action_scores["overall_condition_improvement_score"],
    1.65,
)
assert np.isclose(
    test_overall_action_scores[
        "overall_condition_improvement_score_normalized"
    ],
    1.65 / test_condition_improvement_denominator,
)
assert np.isclose(
    test_overall_action_scores["overall_limiting_factor_amelioration_score"],
    1.85625,
)
assert np.isclose(
    test_overall_action_scores[
        "overall_limiting_factor_amelioration_score_normalized"
    ],
    1.85625 / test_amelioration_denominator,
)
assert np.isclose(
    test_overall_action_scores["overall_benefit_score"],
    0.504,
)
assert np.isclose(
    test_overall_action_scores["overall_benefit_score_normalized"],
    0.504 / test_benefit_denominator,
)

print(
    "All schema, transformation, join, balance, normalization, output, "
    "and equation checks passed."
)
display(test_life)
display(test_lf)
display(test_actions)
display(test_overall_action_scores.to_frame("score"))
